In [ ]:
# Add parent directory to path
import sys
sys.path.append('..')

# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Import custom modules
from src.data.data_generator import DataGenerator, DataProcessor, DataAugmentation
from src.models.transfer_learning import TransferLearningModel
from src.models.ensemble import EnsembleTrainer
from src.utils.evaluation import ModelEvaluator
from src.utils.visualization import ResultVisualizer
from config import *

# Set random seeds for reproducibility
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

# Configure plot style
plt.style.use('seaborn')

In [ ]:
def analyze_dataset(data_dir):
    """Analyze dataset distribution and properties"""
    class_counts = {}
    image_sizes = []
    
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
            
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        class_counts[class_name] = len(images)
        
        # Analyze image sizes
        for img_name in images[:10]:  # Sample first 10 images
            img_path = os.path.join(class_dir, img_name)
            img = plt.imread(img_path)
            image_sizes.append(img.shape)
    
    return class_counts, image_sizes

# Analyze dataset
class_counts, image_sizes = analyze_dataset(DATA_DIR)

# Plot class distribution
plt.figure(figsize=(12, 6))
plt.bar(class_counts.keys(), class_counts.values())
plt.xticks(rotation=45, ha='right')
plt.title('Class Distribution in Dataset')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.show()

# Print image size statistics
print("\nImage Size Statistics:")
sizes_df = pd.DataFrame(image_sizes, columns=['Height', 'Width', 'Channels'])
print(sizes_df.describe())

In [ ]:
def plot_sample_images(data_dir, samples_per_class=3):
    """Plot sample images from each class"""
    fig, axes = plt.subplots(len(CLASS_NAMES), samples_per_class, 
                            figsize=(15, 3*len(CLASS_NAMES)))
    
    for i, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
            
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:samples_per_class]
        
        for j, img_name in enumerate(images):
            img_path = os.path.join(class_dir, img_name)
            img = plt.imread(img_path)
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            
            if j == 0:
                axes[i, j].set_title(f'{class_name}', loc='left')
    
    plt.tight_layout()
    plt.show()

# Plot sample images
plot_sample_images(DATA_DIR)

In [ ]:
def visualize_augmentations(image_path, num_augmentations=5):
    """Visualize different augmentations for a single image"""
    augmenter = DataAugmentation()
    original = plt.imread(image_path)
    
    fig, axes = plt.subplots(1, num_augmentations + 1, figsize=(20, 4))
    
    # Plot original
    axes[0].imshow(original)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    # Plot augmentations
    for i in range(num_augmentations):
        augmented = augmenter.augment(original)
        axes[i+1].imshow(augmented)
        axes[i+1].set_title(f'Augmentation {i+1}')
        axes[i+1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Choose a sample image for augmentation visualization
sample_class = CLASS_NAMES[0]
sample_image = os.listdir(os.path.join(DATA_DIR, sample_class))[0]
sample_path = os.path.join(DATA_DIR, sample_class, sample_image)
visualize_augmentations(sample_path)

In [ ]:
def prepare_data():
    """Prepare data generators for training"""
    # Get all image paths and labels
    image_paths = []
    labels = []
    
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(DATA_DIR, class_name)
        if not os.path.isdir(class_dir):
            continue
            
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(class_dir, img_name))
                labels.append(class_idx)
    
    # Split dataset
    train_paths, test_paths, train_labels, test_labels = train_test_split(
        image_paths, labels, test_size=0.2, stratify=labels, random_state=42
    )
    
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        train_paths, train_labels, test_size=0.2, stratify=train_labels, random_state=42
    )
    
    # Create data generators
    train_generator = DataGenerator(
        train_paths, train_labels,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        augment=True
    )
    
    val_generator = DataGenerator(
        val_paths, val_labels,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        augment=False
    )
    
    test_generator = DataGenerator(
        test_paths, test_labels,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        augment=False
    )
    
    return train_generator, val_generator, test_generator

# Prepare data generators
train_generator, val_generator, test_generator = prepare_data()

In [ ]:
# Initialize transfer learning models
transfer_learning = TransferLearningModel(
    input_shape=(*IMAGE_SIZE, 3),
    num_classes=len(CLASS_NAMES)
)

# Create ensemble
print("Creating ensemble models...")
ensemble_models = transfer_learning.create_ensemble()

# Display model architectures
for i, model in enumerate(ensemble_models):
    print(f"\nModel {i+1} Summary:")
    model.summary()

In [ ]:
# Initialize trainer
trainer = EnsembleTrainer(ensemble_models, batch_size=BATCH_SIZE)

# Train ensemble
print("Training ensemble models...")
histories = trainer.train_ensemble(train_generator, val_generator)

# Save models
for i, model in enumerate(ensemble_models):
    model.save(os.path.join(SAVED_MODELS_DIR, f'model_{i+1}.h5'))

In [ ]:
def plot_training_history(histories):
    """Plot training history for all models"""
    metrics = ['accuracy', 'loss']
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    for i, metric in enumerate(metrics):
        for j, history in enumerate(histories):
            axes[i].plot(history.history[metric], 
                        label=f'Model {j+1} Training')
            axes[i].plot(history.history[f'val_{metric}'], 
                        label=f'Model {j+1} Validation')
        
        axes[i].set_title(f'Model {metric.capitalize()}')
        axes[i].set_xlabel('Epoch')
        axes[i].set_ylabel(metric.capitalize())
        axes[i].legend()
    
    plt.tight_layout()
    plt.show()

# Plot training history
plot_training_history(histories)

In [ ]:
# Initialize evaluator and visualizer
evaluator = ModelEvaluator(ensemble_models[0], CLASS_NAMES)
visualizer = ResultVisualizer(RESULTS_DIR)

# Evaluate models
print("Evaluating ensemble performance...")
conf_matrix, class_metrics, y_pred_probs, y_true = evaluator.calculate_metrics(test_generator)

# Print metrics
print("\nPer-class Metrics:")
for class_name, metrics in class_metrics.items():
    print(f"\n{class_name}:")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")

# Plot confusion matrix
visualizer.plot_confusion_matrix(conf_matrix, CLASS_NAMES)

# Plot ROC curves
visualizer.plot_roc_curves(y_true, y_pred_probs, CLASS_NAMES)

In [ ]:
def visualize_predictions(model, test_generator, num_samples=5):
    """Visualize sample predictions"""
    # Get sample images and predictions
    images, labels = next(iter(test_generator))
    predictions = model.predict(images[:num_samples])
    
    fig, axes = plt.subplots(1, num_samples, figsize=(20, 4))
    
    for i in range(num_samples):
        # Plot image
        axes[i].imshow(images[i])
        axes[i].axis('off')
        
        # Get true and predicted classes
        true_class = CLASS_NAMES[np.argmax(labels[i])]
        pred_class = CLASS_NAMES[np.argmax(predictions[i])]
        
        # Set title color based on prediction correctness
        color = 'green' if true_class == pred_class else 'red'
        axes[i].set_title(f'True: {true_class}\nPred: {pred_class}', 
                         color=color)
    
    plt.tight_layout()
    plt.show()

# Visualize sample predictions
visualize_predictions(ensemble_models[0], test_generator)